# Selective Recall — End-to-End Experiment

This notebook implements and tests **Selective Recall**: a linear-time state-space
backbone with a *gated* attention pathway. A small learned gate decides, per token,
whether that position needs exact recall; only those tokens attend. It is compared
against a pure attention model, a pure state-space model, and a fixed hybrid.

**Nothing is downloaded.** The task is generated in memory. The task is *Selective
Copy with Filler*: a few content tokens must be recalled from before a long stretch
of predictable filler. Only the query positions need recall, so recall is sparse.

**What you should see after running:**
- **pure attention** solves the task (retrieves the earlier token directly).
- **pure state-space** fails as the number of content tokens grows (its fixed state
  cannot hold them all).
- **Selective Recall** matches attention while attending on only a small, learned
  fraction of tokens.
- **oracle gate diagnostic**: the gate fires almost exactly on the recall-critical
  query positions (high precision and recall against the ground-truth labels).

**Runtime.** On a Colab **T4 GPU** the full run takes a few minutes. On CPU it works
but is slower; reduce `CFG.steps` for a quick check.

> **How to run:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
> Run the cells top to bottom.

*Note on results:* these are real numbers from your run. They will differ from the
illustrative numbers in the manuscript. The point is the qualitative pattern.


## 1. Setup
Standard libraries only. All are preinstalled on Colab.

In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("No GPU detected. It will still run; set CFG.steps lower for speed,")
    print("or enable a GPU via Runtime -> Change runtime type -> T4 GPU.")

## 2. Configuration
All knobs live here. The defaults reproduce the reported pattern on a T4 GPU.

In [ ]:
class CFG:
    # task
    K = 8            # content tokens to recall (raise to make the SSM fail harder)
    gap = 3          # filler tokens between queries (controls sparsity)
    n_content = 40   # content alphabet size
    batch = 32

    # model
    d_model = 96
    n_layers = 2
    n_heads = 4
    d_state = 8      # SSM state size (small -> saturates sooner, exposing the gap)

    # training
    steps = 1500     # GPU: 1500 is quick. CPU smoke run: set ~300.
    lr = 2e-3
    weight_decay = 0.0
    grad_clip = 1.0
    phase1_frac = 0.4   # fraction of steps with the gate forced open
    rho = 0.20          # target attending fraction (>= intrinsic recall density)
    lambda_max = 2.0    # peak budget-penalty weight

    # evaluation
    eval_batches = 8
    seed = 0


# token ids: 0=PAD, 1=SEP, 2=QUERY, 3=FILL, content = 4 .. 4+n_content-1
def seq_len(cfg):
    return (cfg.K + 1) + cfg.K * (cfg.gap + 2) - 1

def vocab_size(cfg):
    return 4 + cfg.n_content

## 3. The task: Selective Copy with Filler
Content tokens at the start must be recalled at sparse query positions after a run of filler. `recall_mask` marks the positions that genuinely need recall (the oracle label used later).

In [ ]:
def make_batch(cfg, device):
    """Return input_ids, target_ids, recall_mask, each (batch, L).

    target_ids is -100 except at query positions (loss only there).
    recall_mask is True exactly at query positions (the oracle label).
    """
    L = seq_len(cfg)
    B = cfg.batch
    C0 = 4
    full = np.full((B, L + 1), 3, dtype=np.int64)      # default FILL
    rec = np.zeros((B, L + 1), dtype=bool)
    for b in range(B):
        content = np.random.randint(C0, C0 + cfg.n_content, size=cfg.K)
        seq = list(content) + [1]                       # content block + SEP
        for j in range(cfg.K):
            seq += [3] * cfg.gap + [2, int(content[j])] # filler, QUERY, value
        full[b, :len(seq)] = seq
        pos = cfg.K + 1
        for j in range(cfg.K):
            rec[b, pos + cfg.gap] = True                # mark the QUERY position
            pos += cfg.gap + 2
    full = torch.from_numpy(full).to(device)
    rec = torch.from_numpy(rec).to(device)
    input_ids = full[:, :-1].contiguous()
    target_ids = full[:, 1:].clone()                    # LM shift
    recall_mask = rec[:, :-1].contiguous()
    target_ids[~recall_mask] = -100                     # loss only at queries
    return input_ids, target_ids, recall_mask

A quick look at one example so the layout is clear:

In [ ]:
cfg = CFG()
ids, tgt, rmask = make_batch(cfg, device)
print("sequence length L =", seq_len(cfg), "| vocab =", vocab_size(cfg))
print("recall density (fraction of positions needing recall) =",
      round(rmask.float().mean().item(), 3))
print()
print("tokens 0=PAD 1=SEP 2=QUERY 3=FILL, content>=4")
print("input   :", ids[0].tolist())
tt = tgt[0].tolist()
print("targets :", [t if t != -100 else '.' for t in tt], " ('.' = no loss)")
print("recall  :", [int(x) for x in rmask[0].tolist()])

## 4. Model components
A compact selective state-space layer (Mamba-style S6 core, exact sequential scan), causal multi-head attention, the per-token recall gate, the straight-through estimator for the discrete decision, and the block that fuses them.

In [ ]:
class SelectiveSSM(nn.Module):
    """Compact selective state-space layer (Mamba-style S6 core).
    Diagonal, input-dependent recurrence with a sequential scan. Exact and
    readable; not the CUDA hardware scan. Order is carried by the recurrence."""
    def __init__(self, d_model, d_state=8, d_conv=3):
        super().__init__()
        self.d_model, self.d_state = d_model, d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model)
        self.conv1d = nn.Conv1d(d_model, d_model, d_conv,
                                groups=d_model, padding=d_conv - 1)
        self.dt_proj = nn.Linear(d_model, d_model)
        self.B_proj = nn.Linear(d_model, d_state)
        self.C_proj = nn.Linear(d_model, d_state)
        A_init = torch.log(torch.arange(1, d_state + 1, dtype=torch.float32))
        self.A_log = nn.Parameter(A_init.repeat(d_model, 1))
        self.D = nn.Parameter(torch.ones(d_model))
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B_, L, _ = x.shape
        xin, z = self.in_proj(x).chunk(2, dim=-1)
        xc = self.conv1d(xin.transpose(1, 2))[..., :L].transpose(1, 2)
        xc = F.silu(xc)
        dt = F.softplus(self.dt_proj(xc))          # (B,L,D)
        Bt = self.B_proj(xc)                       # (B,L,N)
        Ct = self.C_proj(xc)                       # (B,L,N)
        A = -torch.exp(self.A_log)                 # (D,N)
        h = x.new_zeros(B_, self.d_model, self.d_state)
        ys = []
        for t in range(L):
            dt_t = dt[:, t, :]
            A_bar = torch.exp(dt_t.unsqueeze(-1) * A.unsqueeze(0))
            Bx = dt_t.unsqueeze(-1) * Bt[:, t].unsqueeze(1) * xc[:, t].unsqueeze(-1)
            h = A_bar * h + Bx
            ys.append((h * Ct[:, t].unsqueeze(1)).sum(-1))
        y = torch.stack(ys, dim=1)
        y = y + xc * self.D
        y = y * F.silu(z)
        return self.out_proj(y)


class CausalMHA(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.dk = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, L, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, L, self.h, self.dk).transpose(1, 2)
        k = k.view(B, L, self.h, self.dk).transpose(1, 2)
        v = v.view(B, L, self.h, self.dk).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(out.transpose(1, 2).contiguous().view(B, L, D))


class RecallGate(nn.Module):
    def __init__(self, d_model, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * d_model, hidden), nn.GELU(), nn.Linear(hidden, 1))

    def forward(self, x, state_readout):
        return torch.sigmoid(self.net(torch.cat([x, state_readout], dim=-1)))


def straight_through(soft):
    """Forward = hard 0/1; backward = smooth sigmoid gradient."""
    hard = (soft > 0.5).float()
    return hard + soft - soft.detach()


class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_state, kind):
        super().__init__()
        self.kind = kind
        self.norm1 = nn.LayerNorm(d_model)
        if kind in ("ssm", "selrec"):
            self.ssm = SelectiveSSM(d_model, d_state)
        if kind in ("attn", "selrec"):
            self.attn = CausalMHA(d_model, n_heads)
        if kind == "selrec":
            self.gate = RecallGate(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.GELU(),
            nn.Linear(4 * d_model, d_model))

    def forward(self, x, gate_open=False):
        h = self.norm1(x)
        g_out = None
        if self.kind == "ssm":
            mix = self.ssm(h)
        elif self.kind == "attn":
            mix = self.attn(h)
        else:  # selrec
            y_ssm = self.ssm(h)
            y_attn = self.attn(h)
            g = self.gate(h, y_ssm)
            g_eff = torch.ones_like(g) if gate_open else straight_through(g)
            mix = y_ssm + g_eff * y_attn
            g_out = g
        x = x + mix
        x = x + self.ffn(self.norm2(x))
        return x, g_out


class SeqModel(nn.Module):
    def __init__(self, vocab, d_model, kinds, n_heads, d_state, max_len):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.pos = nn.Parameter(0.02 * torch.randn(1, max_len, d_model))
        self.blocks = nn.ModuleList(
            [Block(d_model, n_heads, d_state, k) for k in kinds])
        self.norm_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab)

    def forward(self, ids, gate_open=False, return_gate=False):
        x = self.emb(ids) + self.pos[:, :ids.size(1)]
        gates = []
        for blk in self.blocks:
            x, g = blk(x, gate_open=gate_open)
            if g is not None:
                gates.append(g)
        logits = self.head(self.norm_f(x))
        if return_gate:
            gate = torch.stack(gates, 0).mean(0) if gates else None
            return logits, gate
        return logits


def build(kind_name, cfg):
    n = cfg.n_layers
    if kind_name == "pure_attention":
        kinds = ["attn"] * n
    elif kind_name == "pure_ssm":
        kinds = ["ssm"] * n
    elif kind_name == "fixed_hybrid":
        kinds = ["ssm"] * (n - 1) + ["attn"]     # attention at the last layer
    elif kind_name == "selective_recall":
        kinds = ["selrec"] * n
    else:
        raise ValueError(kind_name)
    return SeqModel(vocab_size(cfg), cfg.d_model, kinds,
                    cfg.n_heads, cfg.d_state, seq_len(cfg))

## 5. Training and evaluation
Two-phase schedule: the gate is held open at first so the attention pathway becomes useful, then a one-sided budget penalty anneals in to make the gate sparse. Evaluation reports accuracy, the learned attending fraction, and the oracle gate precision and recall.

In [ ]:
def train_model(kind_name, cfg, device, verbose=False):
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    model = build(kind_name, cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                            weight_decay=cfg.weight_decay)
    p1 = int(cfg.phase1_frac * cfg.steps)
    warmup = max(1, int(0.1 * cfg.steps))

    def lr_at(step):
        if step < warmup:
            return cfg.lr * step / warmup
        prog = (step - warmup) / max(1, cfg.steps - warmup)
        return cfg.lr * 0.5 * (1 + math.cos(math.pi * prog))

    model.train()
    for step in range(cfg.steps):
        for pg in opt.param_groups:
            pg["lr"] = lr_at(step)
        ids, tgt, _ = make_batch(cfg, device)
        gate_open = step < p1
        logits, gate = model(ids, gate_open=gate_open, return_gate=True)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)), tgt.reshape(-1),
            ignore_index=-100)
        if (gate is not None) and (not gate_open):
            lam = cfg.lambda_max * (step - p1) / max(1, cfg.steps - p1)
            loss = loss + lam * F.relu(gate.mean() - cfg.rho)   # one-sided budget
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()
        if verbose and step % max(1, cfg.steps // 6) == 0:
            print(f"    [{kind_name}] step {step:4d} loss {loss.item():.3f}")
    return model


@torch.no_grad()
def evaluate(model, cfg, device):
    model.eval()
    accs, fracs, precs, recs = [], [], [], []
    density = None
    for _ in range(cfg.eval_batches):
        ids, tgt, rmask = make_batch(cfg, device)
        logits, gate = model(ids, gate_open=False, return_gate=True)
        pred = logits.argmax(-1)
        m = tgt != -100
        accs.append((pred[m] == tgt[m]).float().mean().item())
        if density is None:
            density = rmask.float().mean().item()
        if gate is not None:
            fires = gate.squeeze(-1) > 0.5
            fracs.append(fires.float().mean().item())
            tp = (fires & rmask).sum().float()
            precs.append((tp / fires.sum().clamp(min=1)).item())
            recs.append((tp / rmask.sum().clamp(min=1)).item())
    out = {"accuracy": float(np.mean(accs)), "recall_density": density}
    if fracs:
        out.update(attend_frac=float(np.mean(fracs)),
                   gate_precision=float(np.mean(precs)),
                   gate_recall=float(np.mean(recs)))
    return out

## 6. Experiment drivers and plots

In [ ]:
def run_all(cfg, device, verbose=False):
    """Train and evaluate all four models; return a results dict."""
    names = ["pure_attention", "pure_ssm", "fixed_hybrid", "selective_recall"]
    results = {}
    for name in names:
        t0 = time.time()
        model = train_model(name, cfg, device, verbose=verbose)
        res = evaluate(model, cfg, device)
        res["seconds"] = time.time() - t0
        results[name] = res
        if name == "selective_recall":
            results["_sr_model"] = model      # keep for the gate plot
        print(f"  trained {name:18s} acc={res['accuracy']:.3f} "
              f"({res['seconds']:.0f}s)")
    return results


def print_table(results):
    print("\n" + "=" * 78)
    print(f"{'model':20s}{'accuracy':>10s}{'attend_frac':>13s}"
          f"{'gate_P':>9s}{'gate_R':>9s}{'sec':>7s}")
    print("-" * 78)
    for name in ["pure_attention", "pure_ssm", "fixed_hybrid", "selective_recall"]:
        r = results[name]
        af = r.get("attend_frac")
        gp = r.get("gate_precision")
        gr = r.get("gate_recall")
        af_s = f"{af:.3f}" if af is not None else ("1.000" if name == "pure_attention" else "-")
        print(f"{name:20s}{r['accuracy']:>10.3f}{af_s:>13s}"
              f"{(f'{gp:.3f}' if gp is not None else '-'):>9s}"
              f"{(f'{gr:.3f}' if gr is not None else '-'):>9s}"
              f"{r['seconds']:>7.0f}")
    print("=" * 78)
    print(f"(task recall density = "
          f"{results['pure_attention']['recall_density']:.3f}; "
          f"a random gate would score about this precision)")


def difficulty_sweep(cfg, device, K_values=(2, 4, 6, 8), steps=None):
    """Accuracy vs recall load (number of content tokens K) for SSM vs attention
    vs Selective Recall. Reproduces the recall-gap curve."""
    import copy
    out = {"K": list(K_values), "pure_ssm": [], "pure_attention": [],
           "selective_recall": []}
    for K in K_values:
        c = copy.copy(cfg)
        c.K = K
        if steps is not None:
            c.steps = steps
        for name in ["pure_ssm", "pure_attention", "selective_recall"]:
            m = train_model(name, c, device)
            out[name].append(evaluate(m, c, device)["accuracy"])
        print(f"  K={K:2d}  ssm={out['pure_ssm'][-1]:.3f}  "
              f"attn={out['pure_attention'][-1]:.3f}  "
              f"sr={out['selective_recall'][-1]:.3f}")
    return out

In [ ]:
def plot_gate_activation(sr_model, cfg, device, path="fig_gate_activation.png"):
    """Gate probability across positions for a few examples; query positions
    (recall-critical) are marked with dashed lines."""
    import matplotlib.pyplot as plt
    sr_model.eval()
    with torch.no_grad():
        ids, tgt, rmask = make_batch(cfg, device)
        _, gate = sr_model(ids, gate_open=False, return_gate=True)
    g = gate.squeeze(-1).cpu().numpy()
    r = rmask.cpu().numpy()
    n_show = min(4, g.shape[0])
    fig, axes = plt.subplots(n_show, 1, figsize=(9, 1.5 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]
    for i in range(n_show):
        axes[i].bar(np.arange(g.shape[1]), g[i], color="#6a51a3", width=0.9)
        for q in np.where(r[i])[0]:
            axes[i].axvline(q, color="#d95f0e", lw=1.0, ls="--", alpha=0.8)
        axes[i].set_ylim(0, 1.05)
        axes[i].set_ylabel(f"ex {i}")
    axes[-1].set_xlabel("token position  (dashed orange = recall-critical query)")
    axes[0].set_title("Learned gate fires on recall-critical positions")
    plt.tight_layout()
    plt.savefig(path, dpi=140)
    plt.show()
    print("saved", path)


def plot_frontier(results, cfg, path="fig_frontier.png"):
    """Accuracy vs a simple attention-cost proxy for the four models."""
    import matplotlib.pyplot as plt
    n = cfg.n_layers
    cost = {"pure_ssm": 0.0, "pure_attention": 1.0,
            "fixed_hybrid": 1.0 / n,
            "selective_recall": results["selective_recall"].get("attend_frac", 0.2)}
    color = {"pure_ssm": "#2c7fb8", "pure_attention": "#4d4d4d",
             "fixed_hybrid": "#d95f0e", "selective_recall": "#6a51a3"}
    plt.figure(figsize=(6.2, 4.4))
    for name in cost:
        plt.scatter(cost[name], results[name]["accuracy"], s=90,
                    color=color[name], zorder=3)
        plt.annotate(name, (cost[name], results[name]["accuracy"]),
                     textcoords="offset points", xytext=(6, 6), fontsize=8)
    plt.xlabel("relative attention cost  (fraction of tokens/layers attending)")
    plt.ylabel("recall task accuracy")
    plt.ylim(0, 1.05)
    plt.title("Accuracy vs attention cost")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=140)
    plt.show()
    print("saved", path)


def plot_sweep(sweep, path="fig_sweep.png"):
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6.2, 4.4))
    plt.plot(sweep["K"], sweep["pure_attention"], "o-", color="#4d4d4d",
             label="pure attention")
    plt.plot(sweep["K"], sweep["pure_ssm"], "s-", color="#2c7fb8",
             label="pure state-space")
    plt.plot(sweep["K"], sweep["selective_recall"], "D-", color="#6a51a3",
             label="Selective Recall (ours)")
    plt.xlabel("recall load  (number of content tokens K)")
    plt.ylabel("accuracy")
    plt.ylim(0, 1.05)
    plt.title("Recall gap grows with load; Selective Recall tracks attention")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=140)
    plt.show()
    print("saved", path)

## 7. Run the main experiment
Trains all four models on the same data and budget, then prints the comparison table. On a T4 this is a few minutes; the state-space and Selective Recall models are slowest because of the sequential scan.

In [ ]:
cfg = CFG()
# GPU default is fine. For a fast CPU check, uncomment the next line:
# cfg.steps = 400
results = run_all(cfg, device)
print_table(results)

## 8. Oracle gate diagnostic (the key figure)
The gate probability across positions for a few examples. Dashed orange lines mark the recall-critical query positions. A working gate spikes on those and stays near zero elsewhere.

In [ ]:
plot_gate_activation(results["_sr_model"], cfg, device)

## 9. Accuracy vs attention cost
Where each model sits on the accuracy/cost trade-off. Selective Recall should reach attention-level accuracy at a small attending fraction.

In [ ]:
plot_frontier(results, cfg)

## 10. The recall gap vs load
As the number of content tokens `K` grows, the fixed-state model degrades while attention holds. Selective Recall tracks attention. This is the core motivation, reproduced as a curve. This cell trains several models, so it takes a few minutes; reduce `steps` or the `K` list to make it faster.

In [ ]:
sweep = difficulty_sweep(cfg, device, K_values=(2, 4, 6, 8), steps=1000)
plot_sweep(sweep)

## 11. How to read the results, scale up, or use real data

**Reading the table.** `accuracy` is the fraction of query positions answered
correctly. `attend_frac` is the fraction of tokens the gate sent to attention
(only meaningful for Selective Recall). `gate_P` / `gate_R` are the precision and
recall of the gate against the known recall-critical positions. High `gate_R` means
the gate never misses a position that needs recall; high `gate_P` means it wastes
little attention on positions that do not.

**Expected pattern.** pure state-space fails at high load; pure attention solves the
task at full attention cost; Selective Recall matches attention at a fraction of the
attention cost, with the gate firing on the right positions. Your exact numbers will
vary with seed and step count.

**Knobs that matter.**
- `CFG.K` — recall load. Larger `K` makes the state-space model fail harder.
- `CFG.gap` — filler between queries. Larger `gap` makes recall sparser.
- `CFG.d_state` — state-space state size. Larger delays the failure; smaller exposes it.
- `CFG.rho` — target attending fraction. Set it at or slightly above the recall density.
- `CFG.steps` — training length. Attention needs enough steps to solve the task; if
  attention accuracy is low, raise `steps`.

**If the gate collapses** (attending fraction goes to zero and accuracy drops), the
budget pressure rose too early. Increase `phase1_frac`, or lower `lambda_max`.

**Scaling up.** Increase `d_model`, `n_layers`, `K`, and `steps`. The sequential scan
in the state-space layer is the bottleneck on long sequences; for serious runs, swap
`SelectiveSSM` for the official `mamba-ssm` CUDA kernel (same interface: input and
output are `(batch, length, d_model)`).

**Using real text instead of the synthetic task.** Replace `make_batch` with a loader
that yields `(input_ids, target_ids, recall_mask)`. For plain language modeling set
`recall_mask` to all True (every position may need recall) or drop the gate diagnostic;
the four models and the training loop are otherwise unchanged. Tokenize with any
standard tokenizer and stream fixed-length windows.

**Honesty note.** This is a small-scale, single-session study of the *mechanism*, not a
full benchmark. It shows the recall gap, that the gate learns where recall is needed,
and that Selective Recall recovers accuracy cheaply. A full evaluation would add real
corpora, multiple seeds with error bars, and a scaling series, using the same code.
